# Build a Vision-Language Model (VLM) from DeepSeek and OpenAI CLIP

**A modular VLM tutorial example**


This document demonstrates how to transform a text-only large language model (LLM) into a multimodal vision-language model (VLM) using the modular neural network approach that has become mainstream in the last two years.

The notebook includes complete runnable code for the VLM pipeline and was tested in the original environment. After the dataset is installed, it can run directly. The image recognition results shown here come from a model trained for about 30 minutes on the ..... (Supercomputing Internet) platform. The training cost was under CNY 2, using a single RTX 4090 GPU. Other compute platforms can also be used. Depending on your needs, you can choose different LLM scales and CLIP backbones to fit different scenarios.

The core idea is to take a Vision Language Encoder, typically pretrained with CLIP (Contrastive Language-Image Pretraining), and align it through a projector into the embedding space pointed to by the LLM tokenizer. In this way, image content is converted into embedding vectors that a Transformer block can process.

This notebook uses `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` as the LLM and `openai/clip-vit-base-patch32` as the pretrained CLIP vision encoder, then fine-tunes on the Flickr30k dataset to align text and images and train the projector. The dataset contains image-text pairs, and each image has one or five reference captions.

For readers who want to explore related ideas, the following articles may be useful. The embedding space remains an important research direction and has had a deep impact on industry development, but it requires strong mathematical foundations and substantial theory and practice, so it is not usually a central topic in mainstream AI teaching.



## References

1. “Modular Network Assembly and Vision Transform RoPE Experiments: Single Object Tracking on LaSOT”  
   .................................................................................

2. “..... CNN Convolutional Neural Network VGG19 Transfer Learning CIFAR100 Example”  
   .................................................................................

3. “..... Native Training of GPT-2-like Models, 10–50M LLM Example”  
   .................................................................................

All training parameters live inside the projector, while both the CLIP encoder and the LLM are pretrained and frozen. If you want to change that, you can refer to the following examples and further fine-tune them with LoRA or layer-wise unfreezing.

4. “..... Supercomputing Internet LLM Fine-Tuning LoRA Example”  
   .................................................................................

5. “..... Supercomputing Internet LLM Fine-Tuning FSDP LoRA Multi-GPU Distributed Fine-Tuning Example”  
   .................................................................................



Because the image features produced by the CLIP encoder are fed directly into the LLM embedding space through the projector, the model uses a mixed `inputs_embeds` and `input_ids` setup. Therefore, an attention mask must be defined and combined with the model’s causal mask to handle padding tokens. This is reflected in both the `DataLoader` and the training loop.

The LLM uses `float16`, while the CLIP encoder and projector use `float32`. During training, the trainer automatically casts between different precisions, but after training the model may become unusable unless the format conversions are handled carefully inside the model definition. This notebook trains in `float32` and uses `16-bit` during inference to improve speed.

Because the task is text-conditioned, a text prompt must be added to guide the model’s response. The arrangement is `[VISUAL] [PROMPT] [CAPTION]`. Without this prompt, a new VLM may encounter alignment issues because of the LLM’s original behavior. This effect is reduced here because the notebook uses a relatively small LLM.

A typical projector architecture is used. Since the projector contains all trainable parameters, the number of trainable weights is actually small. In practice, meaningful alignment can often be achieved within about five minutes of training, which is one of the common advantages of this method.



## Step 1: Download and preprocess the dataset, then add a prompt


In [ ]:

# -------------------------------
# 1. Environment Setup
# -------------------------------
# !pip install torch torchvision transformers datasets pillow pandas tqdm accelerate
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# -------------------------------
# 2. Load and preprocess Flickr30k
# -------------------------------
import json
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import random

data_root = "/root/private_data/data/flickr30k"
csv_path = os.path.join(data_root, "flickr_annotations_30k.csv")
image_folder = os.path.join(data_root, "flickr30k-images")

df = pd.read_csv(csv_path)
df['captions'] = df['raw'].apply(json.loads)
df['sentids'] = df['sentids'].apply(json.loads)

def is_valid_row(row):
    if len(row['captions']) != 5:
        return False
    img_path = os.path.join(image_folder, row['filename'])
    if not os.path.exists(img_path):
        return False
    return True

df_valid = df[df.apply(is_valid_row, axis=1)].copy()
train_df = df_valid[df_valid['split'] == 'train']
val_df = df_valid[df_valid['split'] == 'val']
test_df = df_valid[df_valid['split'] == 'test']

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

# -------------------------------
# 3. Dataset class WITH PROMPT (fixed padding)
# -------------------------------
class Flickr30kDataset(Dataset):
    def __init__(self, dataframe, image_folder, transform=None, tokenizer=None, max_length=77, prompt="Describe this image:"):
        self.dataframe = dataframe
        self.image_folder = image_folder
        self.transform = transform
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.prompt = prompt

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img_path = os.path.join(self.image_folder, row['filename'])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        captions = row['captions']
        caption = random.choice(captions)

        if self.tokenizer:
            full_text = f"{self.prompt} {caption}"
            tokenized = self.tokenizer(
                full_text,
                truncation=True,
                max_length=self.max_length,
                padding="max_length",
                return_tensors="pt"
            )
            input_ids = tokenized['input_ids'].squeeze(0)
            attention_mask = tokenized['attention_mask'].squeeze(0)

            # Create labels: mask the prompt tokens (including BOS)
            labels = input_ids.clone()
            prompt_tokens = self.tokenizer(self.prompt, add_special_tokens=True)['input_ids']
            prompt_len = len(prompt_tokens)
            labels[:prompt_len] = -100   # ignore loss on prompt

            return {
                'image': image,
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels,
                'caption': caption
            }
        else:
            return {'image': image, 'caption': caption}

# -------------------------------
# 4. Image preprocessing
# -------------------------------
from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Normalize

clip_mean = [0.48145466, 0.4578275, 0.40821073]
clip_std  = [0.26862954, 0.26130258, 0.27577711]

image_transform = Compose([
    Resize(224, interpolation=Image.BICUBIC),
    CenterCrop(224),
    ToTensor(),
    Normalize(mean=clip_mean, std=clip_std),
])



## Step 2: Load the CLIP encoder, LLM, and define the projector


In [ ]:

# -------------------------------
# 5. Load models and tokenizer (fix padding side)
# -------------------------------
from transformers import CLIPVisionModel, AutoModelForCausalLM, AutoTokenizer
import torch.nn as nn

vision_encoder = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
for param in vision_encoder.parameters():
    param.requires_grad = False

llm_model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
llm = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
for param in llm.parameters():
    param.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# CRITICAL: pad on the right so prompt and caption appear at the beginning
tokenizer.padding_side = "right"

# -------------------------------
# 6. VisionProjector
# -------------------------------
class VisionProjector(nn.Module):
    def __init__(self, vision_hidden_size, llm_hidden_size, dropout=0.1):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(vision_hidden_size, llm_hidden_size * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(llm_hidden_size * 2, llm_hidden_size),
        )
    def forward(self, x):
        return self.mlp(x)

# -------------------------------
# 7. VisionLanguageModel (projector in float32, output cast to float16)
# -------------------------------
class VisionLanguageModel(nn.Module):
    def __init__(self, vision_encoder, llm, tokenizer):
        super().__init__()
        self.vision_encoder = vision_encoder
        self.llm = llm
        self.tokenizer = tokenizer
        self.vision_hidden_size = vision_encoder.config.hidden_size
        self.llm_hidden_size = llm.config.hidden_size
        self.projector = VisionProjector(self.vision_hidden_size, self.llm_hidden_size)

    def forward(self, pixel_values, input_ids, attention_mask, labels=None):
        with torch.no_grad():
            vision_outputs = self.vision_encoder(pixel_values=pixel_values)
            vision_features = vision_outputs.last_hidden_state
        visual_embeds = self.projector(vision_features)
        visual_embeds = visual_embeds.to(dtype=self.llm.dtype)

        text_embeds = self.llm.get_input_embeddings()(input_ids)

        inputs_embeds = torch.cat([visual_embeds, text_embeds], dim=1)

        batch_size, num_visual_tokens = visual_embeds.shape[:2]
        visual_attention_mask = torch.ones(batch_size, num_visual_tokens, device=inputs_embeds.device)
        combined_attention_mask = torch.cat([visual_attention_mask, attention_mask], dim=1)

        outputs = self.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=combined_attention_mask,
            return_dict=True
        )
        logits = outputs.logits

        loss = None
        if labels is not None:
            num_vis = visual_embeds.size(1)
            logits_text = logits[:, num_vis:-1, :]
            shift_labels = labels[:, 1:].contiguous()
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits_text.reshape(-1, logits_text.size(-1)), shift_labels.reshape(-1))

        return loss if loss is not None else logits

    def generate(self, pixel_values, max_new_tokens=50, **kwargs):
        with torch.no_grad():
            vision_outputs = self.vision_encoder(pixel_values=pixel_values)
            vision_features = vision_outputs.last_hidden_state
        visual_embeds = self.projector(vision_features)
        visual_embeds = visual_embeds.to(dtype=self.llm.dtype)
        outputs = self.llm.generate(
            inputs_embeds=visual_embeds,
            max_new_tokens=max_new_tokens,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
            **kwargs
        )
        return outputs



## Step 3: Train the projector

Usually, 5–15 epochs are enough to obtain a preliminary result.


In [ ]:

# -------------------------------
# 8. Prepare datasets and dataloaders
# -------------------------------
train_dataset = Flickr30kDataset(
    train_df, image_folder,
    transform=image_transform,
    tokenizer=tokenizer,
    prompt="Describe this image:"
)
val_dataset = Flickr30kDataset(
    val_df, image_folder,
    transform=image_transform,
    tokenizer=tokenizer,
    prompt="Describe this image:"
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)

# -------------------------------
# 9. Initialize model and optimizer
# -------------------------------
vlm = VisionLanguageModel(vision_encoder, llm, tokenizer)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vlm = vlm.to(device)

for name, param in vlm.named_parameters():
    param.requires_grad = False
    if 'projector' in name:
        param.requires_grad = True

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, vlm.parameters()), lr=1e-4)

# -------------------------------
# 10. Training loop with diagnostics (first batch only)
# -------------------------------
from tqdm import tqdm
import torch.cuda.amp as amp

scaler = torch.cuda.amp.GradScaler()
epochs = 15   # train for 15 epochs

diagnostic_printed = False

for epoch in range(epochs):
    vlm.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch_idx, batch in enumerate(progress_bar):
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Print diagnostic info for first batch of first epoch
        if epoch == 0 and batch_idx == 0 and not diagnostic_printed:
            print("\n🔍 DIAGNOSTIC: First training batch")
            print(f"  input_ids shape: {input_ids.shape}")
            print(f"  input_ids[0][:30]: {input_ids[0][:30].tolist()}")
            print(f"  labels[0][:30]: {labels[0][:30].tolist()}")
            first_token = input_ids[0][0].item()
            if first_token == tokenizer.pad_token_id:
                print("  ⚠️ WARNING: First token is PAD! Check tokenizer.padding_side.")
            else:
                print(f"  ✅ First token is {first_token} ({tokenizer.decode([first_token])}) – correct.")
            prompt_tokens = tokenizer("Describe this image:", add_special_tokens=True)['input_ids']
            prompt_len = len(prompt_tokens)
            masked_labels = labels[0][:prompt_len]
            if (masked_labels == -100).all():
                print(f"  ✅ Prompt tokens (first {prompt_len}) are correctly masked (-100).")
            else:
                print(f"  ⚠️ WARNING: Prompt tokens not fully masked.")
            diagnostic_printed = True

        optimizer.zero_grad()
        with amp.autocast():
            loss = vlm(
                pixel_values=images,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Average Training Loss: {avg_loss:.4f}")

    # Validation
    vlm.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            with amp.autocast():
                loss = vlm(
                    pixel_values=images,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch {epoch+1} - Validation Loss: {avg_val_loss:.4f}")



## Step 4: Save the trained model and reload it


In [ ]:

# -------------------------------
# 11. Save projector weights
# -------------------------------
projector_save_path = "projector_epoch15.pth"
torch.save(vlm.projector.state_dict(), projector_save_path)
print(f"Projector saved to {projector_save_path}")

# -------------------------------
# 12. Inference setup (run AFTER training)
# -------------------------------
import torch
import torch.nn as nn
from transformers import CLIPVisionModel, AutoModelForCausalLM, AutoTokenizer
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reload frozen components (or reuse from training)
vision_encoder = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
vision_encoder.eval()
for p in vision_encoder.parameters():
    p.requires_grad = False

llm = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    torch_dtype=torch.float16,
    device_map="auto"
)
llm.eval()
for p in llm.parameters():
    p.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Re-define projector and VLM (same as training, but projector in float16)
class VisionProjector(nn.Module):
    def __init__(self, vision_hidden_size, llm_hidden_size, dropout=0.1):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(vision_hidden_size, llm_hidden_size * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(llm_hidden_size * 2, llm_hidden_size),
        )
    def forward(self, x):
        return self.mlp(x)

class VisionLanguageModel(nn.Module):
    def __init__(self, vision_encoder, llm, tokenizer):
        super().__init__()
        self.vision_encoder = vision_encoder
        self.llm = llm
        self.tokenizer = tokenizer
        self.vision_hidden_size = vision_encoder.config.hidden_size
        self.llm_hidden_size = llm.config.hidden_size
        self.projector = VisionProjector(self.vision_hidden_size, self.llm_hidden_size).to(dtype=llm.dtype)

    def generate(self, pixel_values, max_new_tokens=50, **kwargs):
        with torch.no_grad():
            vision_outputs = self.vision_encoder(pixel_values=pixel_values)
            vision_features = vision_outputs.last_hidden_state
        vision_features = vision_features.to(dtype=self.llm.dtype)
        projected_features = self.projector(vision_features)
        outputs = self.llm.generate(
            inputs_embeds=projected_features,
            max_new_tokens=max_new_tokens,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
            **kwargs
        )
        return outputs





vlm_inference = VisionLanguageModel(vision_encoder, llm, tokenizer).to(device)

# Load projector weights
state_dict = torch.load("projector_epoch15.pth", map_location=device)
for key in state_dict:
    state_dict[key] = state_dict[key].to(dtype=torch.float16)
vlm_inference.projector.load_state_dict(state_dict)
vlm_inference.projector.eval()

print(f"✅ Projector loaded, dtype={vlm_inference.projector.mlp[0].weight.dtype}")

# -------------------------------
# 13. Corrected caption generation function (no slicing)
# -------------------------------
def generate_caption(model, image_tensor, prompt="Describe this image:", max_length=50, temperature=0.7, do_sample=True):
    model.eval()
    with torch.no_grad():
        # Visual embeddings
        vision_outputs = model.vision_encoder(pixel_values=image_tensor.unsqueeze(0).to(device))
        vision_features = vision_outputs.last_hidden_state
        vision_features = vision_features.to(dtype=model.llm.dtype)
        visual_embeds = model.projector(vision_features)

        # Prompt embeddings
        prompt_tokens = model.tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(device)
        prompt_embeds = model.llm.get_input_embeddings()(prompt_tokens.input_ids)

        # Concatenate: visual tokens first, then prompt
        inputs_embeds = torch.cat([visual_embeds, prompt_embeds], dim=1)
        attention_mask = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=device)

        # Generate
        output_ids = model.llm.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_new_tokens=max_length,
            temperature=temperature,
            do_sample=do_sample,
            pad_token_id=model.tokenizer.pad_token_id,
            eos_token_id=model.tokenizer.eos_token_id,
        )
    # output_ids contains ONLY the newly generated tokens (no input tokens)
    return model.tokenizer.decode(output_ids[0], skip_special_tokens=True)



## Step 5: Test on a random sample


In [ ]:

# -------------------------------
# 14. Test on a random validation sample
# -------------------------------
def denormalize(tensor, mean, std):
    tensor = tensor.clone().detach().cpu()
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor.permute(1, 2, 0).numpy()

random_idx = random.randint(0, len(val_dataset) - 1)
sample = val_dataset[random_idx]
image_tensor = sample['image'].to(device)
ground_truth = sample['caption']

generated = generate_caption(vlm_inference, image_tensor, temperature=0.7, do_sample=True)

print(f"Random index: {random_idx}")
print("Ground truth (one of five):", ground_truth)
print("Generated caption:         ", generated)

img_display = denormalize(image_tensor, clip_mean, clip_std)
img_display = np.clip(img_display, 0, 1)

plt.figure(figsize=(8, 8))
plt.imshow(img_display)
plt.axis('off')
plt.title(f"Generated: {generated}\nGround Truth: {ground_truth}", fontsize=12)
plt.tight_layout()
plt.show()



At this point, a Vision-Language Model built from a CLIP encoder and an LLM as modular components is complete.

The example shows that this simple VLM, trained for about 30 minutes, still has limitations, but it does recognize and understand the key content in the image. Because the LLM itself was not fine-tuned, its understanding of the image comes from the newly built VLM’s interpretation of the scene rather than directly from the Flickr30k dataset. Therefore, although the generated response and the ground truth may differ significantly at times, the VLM can still correctly understand the event and intent in the image, and may even be more precise than the training labels themselves.

Note that in this new Vision-Language Model, both the LLM and CLIP are introduced as modular components and are not fine-tuned. For a long time, the Transformer block embedding space in a VLM was considered to be only a text embedding space.

## Contact

Business inquiries: `yucongcai_business@outlook.com`  
Research-related inquiries: `yucongcai_research@outlook.com`
